# Participant-Level Data Integration and Validation

This notebook combines the wearable feature outputs with the cleaned demographic and questionnaire datasets. It verifies that each participant has one final analytical record and creates participant-level training, validation, and test datasets without participant overlap or data leakage.


## Purpose and Expected Outputs

The purpose of this notebook is to:

1. Combine the time-domain, frequency-domain, and wrist-symmetry feature outputs.
2. Aggregate wearable features to one record per participant.
3. Integrate wearable features with cleaned demographic and questionnaire data.
4. Verify participant uniqueness, data completeness, and class distributions.
5. Create participant-level training, validation, and test datasets.
6. Confirm that no participant appears in more than one data split.

The notebook exports:

- `wearable_features.csv`
- `integrated_participant_dataset.csv`
- `train_participant_dataset.csv`
- `validation_participant_dataset.csv`
- `test_participant_dataset.csv`
- `data_integration_validation_summary.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

### Setting the folder paths

In [2]:
# Main data folders
processed_folder = Path("../data/processed")
tables_folder = Path("../outputs/tables")

print("Processed folder exists:", processed_folder.exists())
print("Tables folder exists:", tables_folder.exists())

Processed folder exists: True
Tables folder exists: True


### Setting the datasets paths

In [ ]:
# Cleaned clinical datasets
demographics_file = processed_folder / "demographics_clean.csv"
questionnaire_file = processed_folder / "questionnaire_cleaned.csv"

# Wearable feature datasets
time_features_file = tables_folder / "time_domain_features.csv"
frequency_features_file = tables_folder / "frequency_domain_features.csv"
symmetry_features_file = tables_folder / "wrist_symmetry_features.csv"

In [4]:
required_files = {
    "Demographics": demographics_file,
    "Questionnaire": questionnaire_file,
    "Time-domain features": time_features_file,
    "Frequency-domain features": frequency_features_file,
    "Wrist symmetry features": symmetry_features_file
}

for name, file_path in required_files.items():
    print(f"{name}: {file_path.exists()} — {file_path}")

Demographics: True — ..\data\processed\demographics_clean.csv
Questionnaire: True — ..\data\processed\questionnaire_cleaned.csv
Time-domain features: True — ..\outputs\tables\time_domain_features.csv
Frequency-domain features: True — ..\outputs\tables\frequency_domain_features.csv
Wrist symmetry features: True — ..\outputs\tables\wrist_symmetry_features.csv


### Loading all datasets

In [5]:
demographics = pd.read_csv(demographics_file)
questionnaire = pd.read_csv(questionnaire_file)
time_features = pd.read_csv(time_features_file)
frequency_features = pd.read_csv(frequency_features_file)
symmetry_features = pd.read_csv(symmetry_features_file)

print("All datasets loaded successfully.")

All datasets loaded successfully.


### Checking rows and columns 

In [6]:
print("Demographics shape:", demographics.shape)
print("Questionnaire shape:", questionnaire.shape)
print("Time-domain features shape:", time_features.shape)
print("Frequency-domain features shape:", frequency_features.shape)
print("Symmetry features shape:", symmetry_features.shape)

Demographics shape: (469, 14)
Questionnaire shape: (469, 46)
Time-domain features shape: (10318, 77)
Frequency-domain features shape: (10318, 42)
Symmetry features shape: (5159, 74)


In [7]:
display(demographics.head())
display(questionnaire.head())
display(time_features.head())
display(frequency_features.head())
display(symmetry_features.head())

,patient_id,study_id,condition_original,condition_group,label,age,height_cm,weight_kg,gender,handedness,family_history_any,family_history_first_degree,alcohol_effect_on_tremor,duplicate_patient_id
0,1,PADS,Healthy,Healthy Control,0,56.0,173.0,78.0,Male,Right,Yes,Yes,Unknown,False
1,2,PADS,Other Movement Disorders,Other Movement Disorder,2,81.0,193.0,104.0,Male,Right,No,Unknown,No Effect,False
2,3,PADS,Healthy,Healthy Control,0,45.0,170.0,78.0,Female,Right,No,Unknown,Unknown,False
3,4,PADS,Parkinson's,Parkinson's Disease,1,67.0,161.0,90.0,Female,Right,No,Unknown,No Effect,False
4,5,PADS,Parkinson's,Parkinson's Disease,1,75.0,172.0,86.0,Male,Left,No,Unknown,Unknown,False


,patient_id,questionnaire_name,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08,...,gastrointestinal_count,urinal_count,pain_count,miscellaneous_count,apathy_attention_memory_count,distortion_perception_count,depression_anxiety_count,sexual_function_count,cardiovascular_count,sleep_fatigue_count
0,1,NMS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,NMS,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0,3.0
2,3,NMS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,NMS,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,2.0,1.0,1.0,0.0,1.0,1.0,0.0,2.0,3.0
4,5,NMS,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,5.0


,patient_id,task,wrist,recording_Key,source_File,Accelerometer_X_Mean,Accelerometer_X_Median,Accelerometer_X_Std,Accelerometer_X_Min,Accelerometer_X_Max,...,Acc_Magnitude_Energy,Gyro_Magnitude_Mean,Gyro_Magnitude_Median,Gyro_Magnitude_Std,Gyro_Magnitude_Min,Gyro_Magnitude_Max,Gyro_Magnitude_Range,Gyro_Magnitude_IQR,Gyro_Magnitude_RMS,Gyro_Magnitude_Energy
0,1,CrossArms,LeftWrist,CrossArms__LeftWrist,001_preprocessed.npz,-0.000278,-0.000864,0.086120,-0.405191,0.395058,...,34.294886,1.101626,0.036279,1.931856,0.001287,7.555477,7.554190,1.230665,2.223021,4823.219233
1,1,CrossArms,RightWrist,CrossArms__RightWrist,001_preprocessed.npz,0.000139,-0.000414,0.076206,-0.287730,0.344297,...,26.557016,1.022689,0.038048,1.810530,0.002150,7.615600,7.613449,1.020145,2.078594,4216.858971
2,1,DrinkGlas,LeftWrist,DrinkGlas__LeftWrist,001_preprocessed.npz,0.000321,0.002782,0.063836,-0.212264,0.528248,...,22.423766,0.836871,0.522552,1.022789,0.002307,10.018781,10.016473,1.260728,1.321128,1703.490553
3,1,DrinkGlas,RightWrist,DrinkGlas__RightWrist,001_preprocessed.npz,0.000390,0.000479,0.114052,-0.471591,0.464202,...,54.840312,1.910277,1.134303,2.402229,0.005604,14.187778,14.182174,1.842655,3.068216,9188.015675
4,1,Entrainment,LeftWrist,Entrainment__LeftWrist,001_preprocessed.npz,0.000133,0.001578,0.019979,-0.078619,0.126308,...,4.779990,0.080700,0.068932,0.051170,0.005157,0.369170,0.364013,0.058443,0.095549,18.259263


,patient_id,recording_key,task,wrist,n_samples,original_sampling_frequency_hz,sampling_frequency_hz,resampled_n_samples,uniform_time_start_s,uniform_time_end_s,...,gyro_y_spectral_entropy,gyro_y_spectral_power,gyro_z_dominant_frequency,gyro_z_spectral_centroid,gyro_z_spectral_entropy,gyro_z_spectral_power,gyro_magnitude_dominant_frequency,gyro_magnitude_spectral_centroid,gyro_magnitude_spectral_entropy,gyro_magnitude_spectral_power
0,1,CrossArms__LeftWrist,CrossArms,LeftWrist,976,100.031100,100.0,976,0.0,9.75,...,0.400327,0.202952,0.409836,0.530284,0.372805,0.978077,0.102459,0.354928,0.292612,1.152311
1,1,CrossArms__RightWrist,CrossArms,RightWrist,976,99.344008,100.0,976,0.0,9.75,...,0.380307,0.271797,0.409836,0.538821,0.370770,0.829611,0.102459,0.365697,0.306318,1.201251
2,1,DrinkGlas__LeftWrist,DrinkGlas,LeftWrist,976,100.038257,100.0,976,0.0,9.75,...,0.457844,0.390406,0.102459,0.883988,0.442721,0.334447,0.102459,2.527846,0.629801,1.268809
3,1,DrinkGlas__RightWrist,DrinkGlas,RightWrist,976,99.353420,100.0,976,0.0,9.75,...,0.562906,1.187956,0.102459,1.570955,0.537322,0.564632,0.102459,1.353934,0.522233,1.754532
4,1,Entrainment__LeftWrist,Entrainment,LeftWrist,2000,100.054962,100.0,2000,0.0,19.99,...,0.499449,0.004224,2.150000,2.372372,0.616565,0.002656,0.100000,3.230800,0.664859,0.002213


,patient_id,task,Accelerometer_X_Mean_difference,Accelerometer_X_Median_difference,Accelerometer_X_Std_difference,Accelerometer_X_Min_difference,Accelerometer_X_Max_difference,Accelerometer_X_Range_difference,Accelerometer_X_IQR_difference,Accelerometer_X_RMS_difference,...,Acc_Magnitude_Energy_difference,Gyro_Magnitude_Mean_difference,Gyro_Magnitude_Median_difference,Gyro_Magnitude_Std_difference,Gyro_Magnitude_Min_difference,Gyro_Magnitude_Max_difference,Gyro_Magnitude_Range_difference,Gyro_Magnitude_IQR_difference,Gyro_Magnitude_RMS_difference,Gyro_Magnitude_Energy_difference
0,1,CrossArms,-0.000416,-0.000450,0.009914,-0.117460,0.050761,0.168221,0.002062,0.009909,...,7.737870,0.078937,-0.001769,0.121326,-0.000863,-0.060122,-0.059259,0.210520,0.144427,606.360262
1,1,DrinkGlas,-0.000070,0.002303,-0.050215,0.259327,0.064046,-0.195281,-0.033617,-0.050190,...,-32.416546,-1.073406,-0.611750,-1.379439,-0.003296,-4.168997,-4.165700,-0.581927,-1.747088,-7484.525123
2,1,Entrainment,0.000283,0.001153,-0.001366,0.000176,0.030420,0.030244,-0.000257,-0.001366,...,-1.000469,-0.016382,-0.014885,-0.008695,-0.000212,-0.057450,-0.057239,-0.006170,-0.018499,-7.754731
3,1,HoldWeight,0.000084,0.000432,0.000241,-0.004957,-0.005426,-0.000468,0.000067,0.000243,...,-0.000726,-0.009743,-0.011308,-0.001854,-0.000581,-0.008285,-0.007703,-0.004876,-0.009494,-0.620731
4,1,LiftHold,0.000181,-0.000469,-0.000035,-0.032469,0.007954,0.040423,0.000592,-0.000041,...,-0.605432,-0.022439,-0.000262,-0.007037,0.000818,-0.528812,-0.529629,-0.003706,-0.014988,-18.745048


### Printing all column names

In [8]:
datasets = {
    "Demographics": demographics,
    "Questionnaire": questionnaire,
    "Time-domain features": time_features,
    "Frequency-domain features": frequency_features,
    "Symmetry features": symmetry_features
}

for name, data in datasets.items():
    print(f"\n{name} columns:")
    print(data.columns.tolist())


Demographics columns:
['patient_id', 'study_id', 'condition_original', 'condition_group', 'label', 'age', 'height_cm', 'weight_kg', 'gender', 'handedness', 'family_history_any', 'family_history_first_degree', 'alcohol_effect_on_tremor', 'duplicate_patient_id']

Questionnaire columns:
['patient_id', 'questionnaire_name', 'Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29', 'Q30', 'questions_answered', 'questions_missing', 'questionnaire_status', 'total_symptom_count', 'gastrointestinal_count', 'urinal_count', 'pain_count', 'miscellaneous_count', 'apathy_attention_memory_count', 'distortion_perception_count', 'depression_anxiety_count', 'sexual_function_count', 'cardiovascular_count', 'sleep_fatigue_count']

Time-domain features columns:
['patient_id', 'task', 'wrist', 'recording_Key', 'source_File', 'Accelerometer_X_Mean', 'Accelerometer_

### Checking possible participants ID columns

In [9]:
possible_id_columns = [
    "patient_id",
    "participant_id",
    "subject_id",
    "PatientID",
    "ParticipantID"
]

for name, data in datasets.items():
    matches = [
        column
        for column in possible_id_columns
        if column in data.columns
    ]

    print(f"{name} ID column(s):", matches)

Demographics ID column(s): ['patient_id']
Questionnaire ID column(s): ['patient_id']
Time-domain features ID column(s): ['patient_id']
Frequency-domain features ID column(s): ['patient_id']
Symmetry features ID column(s): ['patient_id']


### Checking repeated participants IDs

In [10]:
for name, data in datasets.items():
    possible_ids = [
        column
        for column in possible_id_columns
        if column in data.columns
    ]

    if possible_ids:
        id_column = possible_ids[0]

        print(f"\n{name}:")
        print("ID column:", id_column)
        print("Rows:", len(data))
        print("Unique IDs:", data[id_column].nunique())
        print("Repeated ID rows:", data[id_column].duplicated().sum())
    else:
        print(f"\n{name}: No recognized ID column found.")


Demographics:
ID column: patient_id
Rows: 469
Unique IDs: 469
Repeated ID rows: 0

Questionnaire:
ID column: patient_id
Rows: 469
Unique IDs: 469
Repeated ID rows: 0

Time-domain features:
ID column: patient_id
Rows: 10318
Unique IDs: 469
Repeated ID rows: 9849

Frequency-domain features:
ID column: patient_id
Rows: 10318
Unique IDs: 469
Repeated ID rows: 9849

Symmetry features:
ID column: patient_id
Rows: 5159
Unique IDs: 469
Repeated ID rows: 4690


In [11]:
datasets = {
    "Demographics": demographics,
    "Questionnaire": questionnaire,
    "Time-domain features": time_features,
    "Frequency-domain features": frequency_features,
    "Symmetry features": symmetry_features
}

### Identifying task, wrist and recording columns

In [12]:
# Checking possible identifier and grouping columns
possible_structure_columns = [
    "patient_id",
    "task",
    "task_name",
    "wrist",
    "side",
    "sensor",
    "recording_id",
    "measurement_id",
    "study_id",
    "condition",
    "label"
]

for name, data in datasets.items():
    found_columns = [
        column
        for column in possible_structure_columns
        if column in data.columns
    ]

    print(f"\n{name}:")
    print("Structure columns found:", found_columns)


Demographics:
Structure columns found: ['patient_id', 'study_id', 'label']

Questionnaire:
Structure columns found: ['patient_id']

Time-domain features:
Structure columns found: ['patient_id', 'task', 'wrist']

Frequency-domain features:
Structure columns found: ['patient_id', 'task', 'wrist']

Symmetry features:
Structure columns found: ['patient_id', 'task']


### Printing the first 15 column names of wearable tables

In [13]:
print("Time-domain columns:")
print(time_features.columns.tolist()[:15])

print("\nFrequency-domain columns:")
print(frequency_features.columns.tolist()[:15])

print("\nSymmetry columns:")
print(symmetry_features.columns.tolist()[:15])

Time-domain columns:
['patient_id', 'task', 'wrist', 'recording_Key', 'source_File', 'Accelerometer_X_Mean', 'Accelerometer_X_Median', 'Accelerometer_X_Std', 'Accelerometer_X_Min', 'Accelerometer_X_Max', 'Accelerometer_X_Range', 'Accelerometer_X_IQR', 'Accelerometer_X_RMS', 'Accelerometer_X_Energy', 'Accelerometer_Y_Mean']

Frequency-domain columns:
['patient_id', 'recording_key', 'task', 'wrist', 'n_samples', 'original_sampling_frequency_hz', 'sampling_frequency_hz', 'resampled_n_samples', 'uniform_time_start_s', 'uniform_time_end_s', 'acc_x_dominant_frequency', 'acc_x_spectral_centroid', 'acc_x_spectral_entropy', 'acc_x_spectral_power', 'acc_y_dominant_frequency']

Symmetry columns:
['patient_id', 'task', 'Accelerometer_X_Mean_difference', 'Accelerometer_X_Median_difference', 'Accelerometer_X_Std_difference', 'Accelerometer_X_Min_difference', 'Accelerometer_X_Max_difference', 'Accelerometer_X_Range_difference', 'Accelerometer_X_IQR_difference', 'Accelerometer_X_RMS_difference', 'Acce

### Showing one participants wearable records

In [14]:
sample_patient_id = time_features["patient_id"].iloc[0]

print("Sample patient ID:", sample_patient_id)

display(
    time_features[
        time_features["patient_id"] == sample_patient_id
    ].head(25)
)

Sample patient ID: 1


,patient_id,task,wrist,recording_Key,source_File,Accelerometer_X_Mean,Accelerometer_X_Median,Accelerometer_X_Std,Accelerometer_X_Min,Accelerometer_X_Max,...,Acc_Magnitude_Energy,Gyro_Magnitude_Mean,Gyro_Magnitude_Median,Gyro_Magnitude_Std,Gyro_Magnitude_Min,Gyro_Magnitude_Max,Gyro_Magnitude_Range,Gyro_Magnitude_IQR,Gyro_Magnitude_RMS,Gyro_Magnitude_Energy
0,1,CrossArms,LeftWrist,CrossArms__LeftWrist,001_preprocessed.npz,-0.000278,-0.000864,0.086120,-0.405191,0.395058,...,34.294886,1.101626,0.036279,1.931856,0.001287,7.555477,7.554190,1.230665,2.223021,4823.219233
1,1,CrossArms,RightWrist,CrossArms__RightWrist,001_preprocessed.npz,0.000139,-0.000414,0.076206,-0.287730,0.344297,...,26.557016,1.022689,0.038048,1.810530,0.002150,7.615600,7.613449,1.020145,2.078594,4216.858971
2,1,DrinkGlas,LeftWrist,DrinkGlas__LeftWrist,001_preprocessed.npz,0.000321,0.002782,0.063836,-0.212264,0.528248,...,22.423766,0.836871,0.522552,1.022789,0.002307,10.018781,10.016473,1.260728,1.321128,1703.490553
3,1,DrinkGlas,RightWrist,DrinkGlas__RightWrist,001_preprocessed.npz,0.000390,0.000479,0.114052,-0.471591,0.464202,...,54.840312,1.910277,1.134303,2.402229,0.005604,14.187778,14.182174,1.842655,3.068216,9188.015675
4,1,Entrainment,LeftWrist,Entrainment__LeftWrist,001_preprocessed.npz,0.000133,0.001578,0.019979,-0.078619,0.126308,...,4.779990,0.080700,0.068932,0.051170,0.005157,0.369170,0.364013,0.058443,0.095549,18.259263
5,1,Entrainment,RightWrist,Entrainment__RightWrist,001_preprocessed.npz,-0.000149,0.000425,0.021345,-0.078794,0.095888,...,5.780458,0.097082,0.083818,0.059865,0.005369,0.426621,0.421252,0.064613,0.114048,26.013994
6,1,HoldWeight,LeftWrist,HoldWeight__LeftWrist,001_preprocessed.npz,0.000143,0.000411,0.003970,-0.015384,0.010517,...,0.070616,0.024835,0.021589,0.014488,0.001762,0.101022,0.099260,0.016429,0.028749,0.806645
7,1,HoldWeight,RightWrist,HoldWeight__RightWrist,001_preprocessed.npz,0.000059,-0.000021,0.003730,-0.010427,0.015943,...,0.071343,0.034579,0.032897,0.016342,0.002344,0.109307,0.106963,0.021306,0.038242,1.427376
8,1,LiftHold,LeftWrist,LiftHold__LeftWrist,001_preprocessed.npz,-0.000554,-0.000604,0.019603,-0.121126,0.069439,...,2.558806,0.230819,0.024352,0.589941,0.002388,2.672846,2.670458,0.030965,0.633207,391.328598
9,1,LiftHold,RightWrist,LiftHold__RightWrist,001_preprocessed.npz,-0.000735,-0.000135,0.019638,-0.088656,0.061485,...,3.164238,0.253258,0.024614,0.596978,0.001570,3.201658,3.200088,0.034671,0.648196,410.073646


In [15]:
display(
    frequency_features[
        frequency_features["patient_id"] == sample_patient_id
    ].head(25)
)

display(
    symmetry_features[
        symmetry_features["patient_id"] == sample_patient_id
    ].head(15)
)

,patient_id,recording_key,task,wrist,n_samples,original_sampling_frequency_hz,sampling_frequency_hz,resampled_n_samples,uniform_time_start_s,uniform_time_end_s,...,gyro_y_spectral_entropy,gyro_y_spectral_power,gyro_z_dominant_frequency,gyro_z_spectral_centroid,gyro_z_spectral_entropy,gyro_z_spectral_power,gyro_magnitude_dominant_frequency,gyro_magnitude_spectral_centroid,gyro_magnitude_spectral_entropy,gyro_magnitude_spectral_power
0,1,CrossArms__LeftWrist,CrossArms,LeftWrist,976,100.031100,100.0,976,0.0,9.75,...,0.400327,0.202952,0.409836,0.530284,0.372805,0.978077,0.102459,0.354928,0.292612,1.152311
1,1,CrossArms__RightWrist,CrossArms,RightWrist,976,99.344008,100.0,976,0.0,9.75,...,0.380307,0.271797,0.409836,0.538821,0.370770,0.829611,0.102459,0.365697,0.306318,1.201251
2,1,DrinkGlas__LeftWrist,DrinkGlas,LeftWrist,976,100.038257,100.0,976,0.0,9.75,...,0.457844,0.390406,0.102459,0.883988,0.442721,0.334447,0.102459,2.527846,0.629801,1.268809
3,1,DrinkGlas__RightWrist,DrinkGlas,RightWrist,976,99.353420,100.0,976,0.0,9.75,...,0.562906,1.187956,0.102459,1.570955,0.537322,0.564632,0.102459,1.353934,0.522233,1.754532
4,1,Entrainment__LeftWrist,Entrainment,LeftWrist,2000,100.054962,100.0,2000,0.0,19.99,...,0.499449,0.004224,2.150000,2.372372,0.616565,0.002656,0.100000,3.230800,0.664859,0.002213
5,1,Entrainment__RightWrist,Entrainment,RightWrist,2000,99.344008,100.0,2000,0.0,19.99,...,0.470653,0.005905,2.000000,2.569225,0.595832,0.002937,0.050000,3.399732,0.686993,0.002839
6,1,HoldWeight__LeftWrist,HoldWeight,LeftWrist,976,100.050189,100.0,976,0.0,9.75,...,0.631547,0.000181,0.922131,2.198491,0.564036,0.000167,0.922131,3.967578,0.676626,0.000125
7,1,HoldWeight__RightWrist,HoldWeight,RightWrist,976,99.311077,100.0,976,0.0,9.75,...,0.649913,0.000349,0.614754,3.508296,0.650107,0.000185,0.307377,5.558891,0.735343,0.000224
8,1,LiftHold__LeftWrist,LiftHold,LeftWrist,976,100.059736,100.0,976,0.0,9.75,...,0.142968,0.001656,0.102459,0.215459,0.124630,0.004490,0.102459,0.173876,0.062838,0.010264
9,1,LiftHold__RightWrist,LiftHold,RightWrist,976,99.364013,100.0,976,0.0,9.75,...,0.331424,0.000919,0.102459,0.204485,0.120683,0.006476,0.102459,0.179711,0.095083,0.016050


,patient_id,task,Accelerometer_X_Mean_difference,Accelerometer_X_Median_difference,Accelerometer_X_Std_difference,Accelerometer_X_Min_difference,Accelerometer_X_Max_difference,Accelerometer_X_Range_difference,Accelerometer_X_IQR_difference,Accelerometer_X_RMS_difference,...,Acc_Magnitude_Energy_difference,Gyro_Magnitude_Mean_difference,Gyro_Magnitude_Median_difference,Gyro_Magnitude_Std_difference,Gyro_Magnitude_Min_difference,Gyro_Magnitude_Max_difference,Gyro_Magnitude_Range_difference,Gyro_Magnitude_IQR_difference,Gyro_Magnitude_RMS_difference,Gyro_Magnitude_Energy_difference
0,1,CrossArms,-0.000416,-0.000450,0.009914,-0.117460,0.050761,0.168221,0.002062,0.009909,...,7.737870,0.078937,-0.001769,0.121326,-0.000863,-0.060122,-0.059259,0.210520,0.144427,606.360262
1,1,DrinkGlas,-0.000070,0.002303,-0.050215,0.259327,0.064046,-0.195281,-0.033617,-0.050190,...,-32.416546,-1.073406,-0.611750,-1.379439,-0.003296,-4.168997,-4.165700,-0.581927,-1.747088,-7484.525123
2,1,Entrainment,0.000283,0.001153,-0.001366,0.000176,0.030420,0.030244,-0.000257,-0.001366,...,-1.000469,-0.016382,-0.014885,-0.008695,-0.000212,-0.057450,-0.057239,-0.006170,-0.018499,-7.754731
3,1,HoldWeight,0.000084,0.000432,0.000241,-0.004957,-0.005426,-0.000468,0.000067,0.000243,...,-0.000726,-0.009743,-0.011308,-0.001854,-0.000581,-0.008285,-0.007703,-0.004876,-0.009494,-0.620731
4,1,LiftHold,0.000181,-0.000469,-0.000035,-0.032469,0.007954,0.040423,0.000592,-0.000041,...,-0.605432,-0.022439,-0.000262,-0.007037,0.000818,-0.528812,-0.529629,-0.003706,-0.014988,-18.745048
5,1,PointFinger,-0.000405,-0.000993,0.082198,0.115594,0.801701,0.686107,0.017818,0.082156,...,68.135733,-0.035693,-0.007643,0.103869,-0.020946,0.966893,0.987840,-0.074567,0.038186,193.667445
6,1,RelaxedTask,0.000060,0.000004,0.002927,-0.014548,0.011034,0.025583,0.003229,0.002925,...,0.138259,0.021880,0.014045,0.022368,-0.000053,0.247708,0.247760,0.024264,0.030976,5.225926
7,1,Relaxed,-0.000039,0.000042,0.000195,0.004461,-0.000469,-0.004930,0.000185,0.000196,...,-0.007514,0.001837,0.001446,0.001454,0.000168,-0.030379,-0.030547,0.001400,0.002343,0.136610
8,1,StretchHold,-0.000195,-0.000182,0.000297,0.000823,0.002894,0.002071,0.000096,0.000300,...,-0.008607,-0.004612,-0.003547,-0.003947,0.001103,-0.046057,-0.047160,-0.004095,-0.005840,-0.299725
9,1,TouchIndex,0.000473,0.000566,-0.008756,0.069007,-0.033304,-0.102311,-0.013218,-0.008754,...,-3.823454,-0.105645,-0.306206,0.088968,-0.008746,0.548725,0.557470,0.340821,-0.036236,-235.463738


## Participant-Level Wearable Feature Aggregation

The time-domain and frequency-domain feature tables contain multiple task and wrist recordings for each participant. These matching recording-level features are first combined using participant, task, and wrist identifiers. Numeric wearable features are then averaged across all available recordings to create one wearable record per participant.

The wrist-symmetry table contains one record for each task. These numeric symmetry features are also averaged across tasks to create one symmetry record per participant.

### Aggregation Strategy

Each participant contains 22 time-domain and frequency-domain records, representing 11 tasks measured on the left and right wrists. Matching records are first combined using `patient_id`, `task`, `wrist`, and `recording_key`.

The numeric time-domain and frequency-domain features are then averaged across the 22 recordings to create one compact wearable representation for each participant. Wrist-symmetry features are averaged across the 11 tasks.

This aggregation produces one wearable record per participant. A limitation is that detailed task-specific and wrist-specific variation is summarized rather than retained as separate columns.

In [16]:
# Standardizing participant IDs
# Converting participant IDs to a consistent integer format
demographics["patient_id"] = pd.to_numeric(
    demographics["patient_id"],
    errors="raise"
).astype(int)

questionnaire["patient_id"] = pd.to_numeric(
    questionnaire["patient_id"],
    errors="raise"
).astype(int)

time_features["patient_id"] = pd.to_numeric(
    time_features["patient_id"],
    errors="raise"
).astype(int)

frequency_features["patient_id"] = pd.to_numeric(
    frequency_features["patient_id"],
    errors="raise"
).astype(int)

symmetry_features["patient_id"] = pd.to_numeric(
    symmetry_features["patient_id"],
    errors="raise"
).astype(int)

print("Participant IDs standardized successfully.")


Participant IDs standardized successfully.


In [17]:
print("Time recording columns:")
print([
    column for column in time_features.columns
    if "recording" in column.lower()
])

print("\nFrequency recording columns:")
print([
    column for column in frequency_features.columns
    if "recording" in column.lower()
])

Time recording columns:
['recording_Key']

Frequency recording columns:
['recording_key']


In [18]:
# Standardizing recording-key column names
time_features = time_features.rename(
    columns={
        "recording_Key": "recording_key",
        "Recording_Key": "recording_key"
    }
)

frequency_features = frequency_features.rename(
    columns={
        "recording_Key": "recording_key",
        "Recording_Key": "recording_key"
    }
)

print(
    "Time recording key exists:",
    "recording_key" in time_features.columns
)

print(
    "Frequency recording key exists:",
    "recording_key" in frequency_features.columns
)

Time recording key exists: True
Frequency recording key exists: True


In [19]:
# Columns used only to describe each recording
time_metadata_columns = [
    "patient_id",
    "task",
    "wrist",
    "recording_key",
    "source_file"
]

frequency_metadata_columns = [
    "patient_id",
    "task",
    "wrist",
    "recording_key"
]

# Select numeric feature columns only
time_numeric_columns = (
    time_features
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

frequency_numeric_columns = (
    frequency_features
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

# patient_id is an identifier, not a wearable feature
if "patient_id" in time_numeric_columns:
    time_numeric_columns.remove("patient_id")

if "patient_id" in frequency_numeric_columns:
    frequency_numeric_columns.remove("patient_id")

print("Time-domain numeric features:", len(time_numeric_columns))
print("Frequency-domain numeric features:", len(frequency_numeric_columns))

Time-domain numeric features: 72
Frequency-domain numeric features: 38


In [20]:
# Keep identifiers plus numeric wearable features
time_for_merge = time_features[
    [
        "patient_id",
        "task",
        "wrist",
        "recording_key"
    ] + time_numeric_columns
].copy()

frequency_for_merge = frequency_features[
    [
        "patient_id",
        "task",
        "wrist",
        "recording_key"
    ] + frequency_numeric_columns
].copy()

# Prefix feature names
time_for_merge = time_for_merge.rename(
    columns={
        column: f"time_{column}"
        for column in time_numeric_columns
    }
)

frequency_for_merge = frequency_for_merge.rename(
    columns={
        column: f"freq_{column}"
        for column in frequency_numeric_columns
    }
)

print("Feature prefixes added successfully.")

Feature prefixes added successfully.


In [21]:
recording_merge_keys = [
    "patient_id",
    "task",
    "wrist",
    "recording_key"
]

print(
    "Duplicate time-domain recording keys:",
    time_for_merge.duplicated(
        subset=recording_merge_keys
    ).sum()
)

print(
    "Duplicate frequency-domain recording keys:",
    frequency_for_merge.duplicated(
        subset=recording_merge_keys
    ).sum()
)

Duplicate time-domain recording keys: 0
Duplicate frequency-domain recording keys: 0


In [22]:
#Combining matching time and frequency recordings
wearable_recordings = time_for_merge.merge(
    frequency_for_merge,
    on=[
        "patient_id",
        "task",
        "wrist",
        "recording_key"
    ],
    how="inner",
    validate="one_to_one"
)

print("Combined wearable recording shape:", wearable_recordings.shape)
print(
    "Unique participants:",
    wearable_recordings["patient_id"].nunique()
)

print(
    "Unique recording keys:",
    wearable_recordings["recording_key"].nunique()
)

Combined wearable recording shape: (10318, 114)
Unique participants: 469
Unique recording keys: 22


In [23]:
print("Original time-domain rows:", len(time_features))
print("Original frequency-domain rows:", len(frequency_features))
print("Matched wearable rows:", len(wearable_recordings))

unmatched_time_rows = len(time_features) - len(wearable_recordings)
unmatched_frequency_rows = (
    len(frequency_features) - len(wearable_recordings)
)

print("Unmatched time-domain rows:", unmatched_time_rows)
print(
    "Unmatched frequency-domain rows:",
    unmatched_frequency_rows
)

Original time-domain rows: 10318
Original frequency-domain rows: 10318
Matched wearable rows: 10318
Unmatched time-domain rows: 0
Unmatched frequency-domain rows: 0


In [24]:
# Selecting all combined numeric wearable features
combined_numeric_columns = (
    wearable_recordings
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

combined_numeric_columns.remove("patient_id")

# Average the 22 task/wrist recordings for each participant
wearable_participant = (
    wearable_recordings
    .groupby("patient_id")[combined_numeric_columns]
    .mean()
    .reset_index()
)

print(
    "Participant-level time/frequency shape:",
    wearable_participant.shape
)

print(
    "Unique participants:",
    wearable_participant["patient_id"].nunique()
)

print(
    "Duplicate participant IDs:",
    wearable_participant["patient_id"].duplicated().sum()
)

display(wearable_participant.head())

Participant-level time/frequency shape: (469, 111)
Unique participants: 469
Duplicate participant IDs: 0


,patient_id,time_Accelerometer_X_Mean,time_Accelerometer_X_Median,time_Accelerometer_X_Std,time_Accelerometer_X_Min,time_Accelerometer_X_Max,time_Accelerometer_X_Range,time_Accelerometer_X_IQR,time_Accelerometer_X_RMS,time_Accelerometer_X_Energy,...,freq_gyro_y_spectral_entropy,freq_gyro_y_spectral_power,freq_gyro_z_dominant_frequency,freq_gyro_z_spectral_centroid,freq_gyro_z_spectral_entropy,freq_gyro_z_spectral_power,freq_gyro_magnitude_dominant_frequency,freq_gyro_magnitude_spectral_centroid,freq_gyro_magnitude_spectral_entropy,freq_gyro_magnitude_spectral_power
0,1,-0.000155,0.004059,0.074081,-0.513615,0.233092,0.746707,0.059540,0.074046,14.549401,...,0.496097,0.982722,1.077794,2.210475,0.466672,1.788369,0.558197,2.891241,0.510830,1.563943
1,2,0.000043,-0.000302,0.062226,-0.365569,0.382510,0.748079,0.057380,0.062207,6.704495,...,0.498420,0.455997,1.992735,2.448173,0.482142,0.353557,0.896013,3.300614,0.536026,0.704772
2,3,-0.000092,-0.000014,0.089180,-0.364048,0.331769,0.695817,0.094730,0.089143,20.022262,...,0.511789,1.428762,1.426565,2.503264,0.432065,4.204214,1.033793,3.088925,0.472244,1.678728
3,4,-0.000061,-0.000984,0.035592,-0.177022,0.220743,0.397765,0.024906,0.035588,4.321860,...,0.408111,0.405129,1.101155,1.652757,0.376839,0.878390,0.406483,1.821555,0.388259,0.476171
4,5,-0.000042,-0.000507,0.060632,-0.213993,0.233428,0.447422,0.075401,0.060610,6.021132,...,0.485392,0.459802,2.552124,2.723202,0.428897,0.744560,0.902012,3.742713,0.531731,0.418009


In [25]:
symmetry_numeric_columns = (
    symmetry_features
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

if "patient_id" in symmetry_numeric_columns:
    symmetry_numeric_columns.remove("patient_id")

# Add a prefix so symmetry columns are easy to identify
symmetry_for_aggregation = symmetry_features[
    ["patient_id"] + symmetry_numeric_columns
].copy()

symmetry_for_aggregation = symmetry_for_aggregation.rename(
    columns={
        column: f"symmetry_{column}"
        for column in symmetry_numeric_columns
    }
)

prefixed_symmetry_columns = [
    f"symmetry_{column}"
    for column in symmetry_numeric_columns
]

symmetry_participant = (
    symmetry_for_aggregation
    .groupby("patient_id")[prefixed_symmetry_columns]
    .mean()
    .reset_index()
)

print(
    "Participant-level symmetry shape:",
    symmetry_participant.shape
)

print(
    "Unique participants:",
    symmetry_participant["patient_id"].nunique()
)

print(
    "Duplicate participant IDs:",
    symmetry_participant["patient_id"].duplicated().sum()
)

display(symmetry_participant.head())

Participant-level symmetry shape: (469, 73)
Unique participants: 469
Duplicate participant IDs: 0


,patient_id,symmetry_Accelerometer_X_Mean_difference,symmetry_Accelerometer_X_Median_difference,symmetry_Accelerometer_X_Std_difference,symmetry_Accelerometer_X_Min_difference,symmetry_Accelerometer_X_Max_difference,symmetry_Accelerometer_X_Range_difference,symmetry_Accelerometer_X_IQR_difference,symmetry_Accelerometer_X_RMS_difference,symmetry_Accelerometer_X_Energy_difference,...,symmetry_Acc_Magnitude_Energy_difference,symmetry_Gyro_Magnitude_Mean_difference,symmetry_Gyro_Magnitude_Median_difference,symmetry_Gyro_Magnitude_Std_difference,symmetry_Gyro_Magnitude_Min_difference,symmetry_Gyro_Magnitude_Max_difference,symmetry_Gyro_Magnitude_Range_difference,symmetry_Gyro_Magnitude_IQR_difference,symmetry_Gyro_Magnitude_RMS_difference,symmetry_Gyro_Magnitude_Energy_difference
0,1,0.000126,0.000751,0.002687,0.020347,0.081354,0.061007,-0.001020,0.002685,3.926234,...,1.631548,-0.126805,-0.085487,-0.123576,-0.007836,-0.317006,-0.309170,-0.028445,-0.179823,-943.554969
1,2,0.000311,0.003689,-0.000609,0.157281,-0.055430,-0.212710,0.003881,-0.000597,-0.070427,...,-11.633377,-0.064486,-0.106319,0.023457,-0.004497,-0.520227,-0.515729,-0.031305,-0.028431,202.427722
2,3,0.000396,-0.001194,0.005026,0.115035,0.146842,0.031807,0.007943,0.005028,2.695090,...,0.987351,0.068282,0.084292,0.060849,0.017704,0.438968,0.421265,0.056464,0.092100,925.543812
3,4,-0.000216,-0.003590,0.010988,0.067240,0.161462,0.094221,0.008039,0.010974,2.495382,...,-2.504678,0.014927,0.036575,0.000743,0.001043,0.170965,0.169922,0.052439,0.012388,-69.792417
4,5,0.000666,0.004864,-0.028622,0.059647,-0.083144,-0.142791,-0.040769,-0.028618,-4.931818,...,-39.516070,-0.404767,-0.398273,-0.190974,-0.010912,-1.140813,-1.129902,-0.373412,-0.428032,-833.061251


In [26]:
wearable_features = wearable_participant.merge(
    symmetry_participant,
    on="patient_id",
    how="inner",
    validate="one_to_one"
)

print("Final wearable feature shape:", wearable_features.shape)

print(
    "Unique participants:",
    wearable_features["patient_id"].nunique()
)

print(
    "Duplicate participant IDs:",
    wearable_features["patient_id"].duplicated().sum()
)

display(wearable_features.head())

Final wearable feature shape: (469, 183)
Unique participants: 469
Duplicate participant IDs: 0


,patient_id,time_Accelerometer_X_Mean,time_Accelerometer_X_Median,time_Accelerometer_X_Std,time_Accelerometer_X_Min,time_Accelerometer_X_Max,time_Accelerometer_X_Range,time_Accelerometer_X_IQR,time_Accelerometer_X_RMS,time_Accelerometer_X_Energy,...,symmetry_Acc_Magnitude_Energy_difference,symmetry_Gyro_Magnitude_Mean_difference,symmetry_Gyro_Magnitude_Median_difference,symmetry_Gyro_Magnitude_Std_difference,symmetry_Gyro_Magnitude_Min_difference,symmetry_Gyro_Magnitude_Max_difference,symmetry_Gyro_Magnitude_Range_difference,symmetry_Gyro_Magnitude_IQR_difference,symmetry_Gyro_Magnitude_RMS_difference,symmetry_Gyro_Magnitude_Energy_difference
0,1,-0.000155,0.004059,0.074081,-0.513615,0.233092,0.746707,0.059540,0.074046,14.549401,...,1.631548,-0.126805,-0.085487,-0.123576,-0.007836,-0.317006,-0.309170,-0.028445,-0.179823,-943.554969
1,2,0.000043,-0.000302,0.062226,-0.365569,0.382510,0.748079,0.057380,0.062207,6.704495,...,-11.633377,-0.064486,-0.106319,0.023457,-0.004497,-0.520227,-0.515729,-0.031305,-0.028431,202.427722
2,3,-0.000092,-0.000014,0.089180,-0.364048,0.331769,0.695817,0.094730,0.089143,20.022262,...,0.987351,0.068282,0.084292,0.060849,0.017704,0.438968,0.421265,0.056464,0.092100,925.543812
3,4,-0.000061,-0.000984,0.035592,-0.177022,0.220743,0.397765,0.024906,0.035588,4.321860,...,-2.504678,0.014927,0.036575,0.000743,0.001043,0.170965,0.169922,0.052439,0.012388,-69.792417
4,5,-0.000042,-0.000507,0.060632,-0.213993,0.233428,0.447422,0.075401,0.060610,6.021132,...,-39.516070,-0.404767,-0.398273,-0.190974,-0.010912,-1.140813,-1.129902,-0.373412,-0.428032,-833.061251


In [27]:
wearable_missing_values = int(
    wearable_features.isna().sum().sum()
)

wearable_numeric_data = wearable_features.select_dtypes(
    include=np.number
)

wearable_infinite_values = int(
    np.isinf(wearable_numeric_data).sum().sum()
)

print(
    "Total missing wearable values:",
    wearable_missing_values
)

print(
    "Total infinite wearable values:",
    wearable_infinite_values
)

Total missing wearable values: 0
Total infinite wearable values: 0


In [28]:
wearable_output_file = (
    processed_folder / "wearable_features.csv"
)

wearable_features.to_csv(
    wearable_output_file,
    index=False
)

print("Wearable feature table saved successfully.")
print("Saved location:", wearable_output_file.resolve())

Wearable feature table saved successfully.
Saved location: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\wearable_features.csv


## Integration with Demographic and Questionnaire Data

The participant-level wearable features are combined with the cleaned demographic and questionnaire datasets using `patient_id`. Validation checks are performed to confirm that each participant has one final analytical record.

### Checking participant overlap across datasets

In [29]:
# Creating participant ID sets
demographic_ids = set(demographics["patient_id"])
questionnaire_ids = set(questionnaire["patient_id"])
wearable_ids = set(wearable_features["patient_id"])

# Participants present in all three datasets
common_ids = (
    demographic_ids
    & questionnaire_ids
    & wearable_ids
)

print("Demographic participants:", len(demographic_ids))
print("Questionnaire participants:", len(questionnaire_ids))
print("Wearable participants:", len(wearable_ids))
print("Participants found in all three datasets:", len(common_ids))

Demographic participants: 469
Questionnaire participants: 469
Wearable participants: 469
Participants found in all three datasets: 469


### Checking for duplicate participant IDs

In [30]:
print(
    "Duplicate IDs in demographics:",
    demographics["patient_id"].duplicated().sum()
)

print(
    "Duplicate IDs in questionnaire:",
    questionnaire["patient_id"].duplicated().sum()
)

print(
    "Duplicate IDs in wearable features:",
    wearable_features["patient_id"].duplicated().sum()
)

Duplicate IDs in demographics: 0
Duplicate IDs in questionnaire: 0
Duplicate IDs in wearable features: 0


### Checking overlapping column names

In [31]:
demographic_columns = set(demographics.columns)
questionnaire_columns = set(questionnaire.columns)

overlapping_columns = (
    demographic_columns
    & questionnaire_columns
) - {"patient_id"}

print("Overlapping columns:")
print(sorted(overlapping_columns))

Overlapping columns:
[]


### Merging demographics and questionnaire

In [32]:
clinical_data = demographics.merge(
    questionnaire,
    on="patient_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_demographic", "_questionnaire")
)

print("Clinical dataset shape:", clinical_data.shape)

print(
    "Unique clinical participants:",
    clinical_data["patient_id"].nunique()
)

print(
    "Duplicate clinical participant IDs:",
    clinical_data["patient_id"].duplicated().sum()
)

display(clinical_data.head())

Clinical dataset shape: (469, 59)
Unique clinical participants: 469
Duplicate clinical participant IDs: 0


,patient_id,study_id,condition_original,condition_group,label,age,height_cm,weight_kg,gender,handedness,...,gastrointestinal_count,urinal_count,pain_count,miscellaneous_count,apathy_attention_memory_count,distortion_perception_count,depression_anxiety_count,sexual_function_count,cardiovascular_count,sleep_fatigue_count
0,1,PADS,Healthy,Healthy Control,0,56.0,173.0,78.0,Male,Right,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,PADS,Other Movement Disorders,Other Movement Disorder,2,81.0,193.0,104.0,Male,Right,...,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0,3.0
2,3,PADS,Healthy,Healthy Control,0,45.0,170.0,78.0,Female,Right,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,PADS,Parkinson's,Parkinson's Disease,1,67.0,161.0,90.0,Female,Right,...,1.0,2.0,1.0,1.0,0.0,1.0,1.0,0.0,2.0,3.0
4,5,PADS,Parkinson's,Parkinson's Disease,1,75.0,172.0,86.0,Male,Left,...,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,5.0


### Merging clinical and wearable data

In [33]:
integrated_data = clinical_data.merge(
    wearable_features,
    on="patient_id",
    how="inner",
    validate="one_to_one"
)

print("Integrated dataset shape:", integrated_data.shape)

print(
    "Unique participants:",
    integrated_data["patient_id"].nunique()
)

print(
    "Duplicate participant IDs:",
    integrated_data["patient_id"].duplicated().sum()
)

display(integrated_data.head())

Integrated dataset shape: (469, 241)
Unique participants: 469
Duplicate participant IDs: 0


,patient_id,study_id,condition_original,condition_group,label,age,height_cm,weight_kg,gender,handedness,...,symmetry_Acc_Magnitude_Energy_difference,symmetry_Gyro_Magnitude_Mean_difference,symmetry_Gyro_Magnitude_Median_difference,symmetry_Gyro_Magnitude_Std_difference,symmetry_Gyro_Magnitude_Min_difference,symmetry_Gyro_Magnitude_Max_difference,symmetry_Gyro_Magnitude_Range_difference,symmetry_Gyro_Magnitude_IQR_difference,symmetry_Gyro_Magnitude_RMS_difference,symmetry_Gyro_Magnitude_Energy_difference
0,1,PADS,Healthy,Healthy Control,0,56.0,173.0,78.0,Male,Right,...,1.631548,-0.126805,-0.085487,-0.123576,-0.007836,-0.317006,-0.309170,-0.028445,-0.179823,-943.554969
1,2,PADS,Other Movement Disorders,Other Movement Disorder,2,81.0,193.0,104.0,Male,Right,...,-11.633377,-0.064486,-0.106319,0.023457,-0.004497,-0.520227,-0.515729,-0.031305,-0.028431,202.427722
2,3,PADS,Healthy,Healthy Control,0,45.0,170.0,78.0,Female,Right,...,0.987351,0.068282,0.084292,0.060849,0.017704,0.438968,0.421265,0.056464,0.092100,925.543812
3,4,PADS,Parkinson's,Parkinson's Disease,1,67.0,161.0,90.0,Female,Right,...,-2.504678,0.014927,0.036575,0.000743,0.001043,0.170965,0.169922,0.052439,0.012388,-69.792417
4,5,PADS,Parkinson's,Parkinson's Disease,1,75.0,172.0,86.0,Male,Left,...,-39.516070,-0.404767,-0.398273,-0.190974,-0.010912,-1.140813,-1.129902,-0.373412,-0.428032,-833.061251


### Verifying one final analytical record per participant

In [34]:
one_record_per_participant = (
    len(integrated_data)
    == integrated_data["patient_id"].nunique()
)

print(
    "One analytical record per participant:",
    one_record_per_participant
)

One analytical record per participant: True


### Checking missing values

In [35]:
missing_summary = (
    integrated_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

columns_with_missing_values = missing_summary[
    missing_summary > 0
]

print(
    "Number of columns with missing values:",
    len(columns_with_missing_values)
)

display(columns_with_missing_values.head(20))

Number of columns with missing values: 1


height_cm    1
dtype: int64

### Missing-Value Interpretation

The final integrated dataset contains 99 missing values across two variables:

- `age_at_diagnosis`: Missing values may be clinically appropriate for participants without a Parkinson’s disease diagnosis.
- `height_cm`: One participant has a missing height value.

No imputation is performed in this integration notebook. To prevent data leakage, any numerical imputation should be fitted using the training dataset only and then applied unchanged to the validation and test datasets during model preprocessing.

### Checking infinite values 

In [36]:
integrated_numeric = integrated_data.select_dtypes(
    include=np.number
)

total_infinite_values = int(
    np.isinf(integrated_numeric).sum().sum()
)

print("Total infinite values:", total_infinite_values)

Total infinite values: 0


In [37]:
assert integrated_data["patient_id"].is_unique, (
    "Each participant must have exactly one analytical record."
)

assert integrated_data["patient_id"].nunique() == 469, (
    "Expected 469 unique participants."
)

assert total_infinite_values == 0, (
    "Infinite values were found in the integrated dataset."
)

print("Core participant-level validation checks passed.")

Core participant-level validation checks passed.


### Checking the diagnosis target 

In [38]:
possible_target_columns = [
    "label",
    "condition_group",
    "condition_original",
    "diagnosis",
    "diagnosis_group",
    "target"
]

available_target_columns = [
    column
    for column in possible_target_columns
    if column in integrated_data.columns
]

print("Available target columns:")
print(available_target_columns)

Available target columns:
['label', 'condition_group', 'condition_original']


### Examining the label distribution

In [39]:
target_column = "label"

print("Target values:")
print(
    integrated_data[target_column]
    .value_counts(dropna=False)
)

print("\nTarget proportions:")
print(
    integrated_data[target_column]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .round(3)
)

Target values:
label
1    276
2    114
0     79
Name: count, dtype: int64

Target proportions:
label
1    0.588
2    0.243
0    0.168
Name: proportion, dtype: float64


In [40]:
print(
    "Missing target values:",
    integrated_data[target_column].isna().sum()
)

Missing target values: 0


### Target-Leakage Consideration

The integrated dataset contains `label`, `condition_group`, and `condition_original`. Because these variables describe the participant's diagnostic outcome, `condition_group` and `condition_original` must not be used as predictors when `label` is the modelling target.

Participant identifiers such as `patient_id` and `study_id` should also be excluded from the model feature matrix. They are retained in the exported datasets only for participant tracking and leakage validation.

In [41]:
# Columns that must not be used as model predictors
non_predictor_columns = [
    "patient_id",
    "study_id",
    "label",
    "condition_group",
    "condition_original"
]

existing_non_predictors = [
    column
    for column in non_predictor_columns
    if column in integrated_data.columns
]

print("Columns excluded from model predictors:")
print(existing_non_predictors)

Columns excluded from model predictors:
['patient_id', 'study_id', 'label', 'condition_group', 'condition_original']


## Participant-Level Train, Validation and Test Split

The integrated dataset is split at the participant level. Because each participant has one row, each participant can appear in only one dataset. Stratification is used to maintain similar diagnosis-class proportions across the training, validation and test sets.

### Creating  the Initial Train-Temporary Split 
This creates:
70% training
30% temporary


In [42]:
train_data, temporary_data = train_test_split(
    integrated_data,
    test_size=0.30,
    random_state=42,
    stratify=integrated_data[target_column]
)

print("Training rows:", len(train_data))
print("Temporary rows:", len(temporary_data))

Training rows: 328
Temporary rows: 141


### Creating validation and test datasets

In [43]:
validation_data, test_data = train_test_split(
    temporary_data,
    test_size=0.50,
    random_state=42,
    stratify=temporary_data[target_column]
)

print("Training rows:", len(train_data))
print("Validation rows:", len(validation_data))
print("Testing rows:", len(test_data))

print(
    "Total rows:",
    len(train_data)
    + len(validation_data)
    + len(test_data)
)

Training rows: 328
Validation rows: 70
Testing rows: 71
Total rows: 469


### Checking participant leakage 

In [44]:
train_ids = set(train_data["patient_id"])
validation_ids = set(validation_data["patient_id"])
test_ids = set(test_data["patient_id"])

train_validation_overlap = len(
    train_ids & validation_ids
)

train_test_overlap = len(
    train_ids & test_ids
)

validation_test_overlap = len(
    validation_ids & test_ids
)

print(
    "Train-validation participant overlap:",
    train_validation_overlap
)

print(
    "Train-test participant overlap:",
    train_test_overlap
)

print(
    "Validation-test participant overlap:",
    validation_test_overlap
)

Train-validation participant overlap: 0
Train-test participant overlap: 0
Validation-test participant overlap: 0


### Confirming every participant is assigned once

In [45]:
all_split_ids = (
    train_ids
    | validation_ids
    | test_ids
)

print(
    "Participants in integrated dataset:",
    integrated_data["patient_id"].nunique()
)

print(
    "Participants assigned across all splits:",
    len(all_split_ids)
)

print(
    "All participants assigned exactly once:",
    len(all_split_ids)
    == integrated_data["patient_id"].nunique()
)

Participants in integrated dataset: 469
Participants assigned across all splits: 469
All participants assigned exactly once: True


In [46]:
assert train_validation_overlap == 0, (
    "Participant overlap found between training and validation."
)

assert train_test_overlap == 0, (
    "Participant overlap found between training and testing."
)

assert validation_test_overlap == 0, (
    "Participant overlap found between validation and testing."
)

assert len(all_split_ids) == integrated_data["patient_id"].nunique(), (
    "Not every participant was assigned exactly once."
)

print("Participant leakage validation passed.")

Participant leakage validation passed.


### Creating a class-distribution function


In [47]:
def show_class_distribution(
    data,
    dataset_name,
    target
):
    print(f"\n{dataset_name} class counts:")

    print(
        data[target]
        .value_counts()
        .sort_index()
    )

    print(f"\n{dataset_name} class proportions:")

    print(
        data[target]
        .value_counts(normalize=True)
        .sort_index()
        .round(3)
    )

### Displaying class distributions

In [48]:
show_class_distribution(
    train_data,
    "Training",
    target_column
)

show_class_distribution(
    validation_data,
    "Validation",
    target_column
)

show_class_distribution(
    test_data,
    "Testing",
    target_column
)


Training class counts:
label
0     55
1    193
2     80
Name: count, dtype: int64

Training class proportions:
label
0    0.168
1    0.588
2    0.244
Name: proportion, dtype: float64

Validation class counts:
label
0    12
1    41
2    17
Name: count, dtype: int64

Validation class proportions:
label
0    0.171
1    0.586
2    0.243
Name: proportion, dtype: float64

Testing class counts:
label
0    12
1    42
2    17
Name: count, dtype: int64

Testing class proportions:
label
0    0.169
1    0.592
2    0.239
Name: proportion, dtype: float64


### Creating the validation summary 

In [49]:
validation_results = {
    "integrated_rows": integrated_data.shape[0],
    "integrated_columns": integrated_data.shape[1],
    "unique_participants": (
        integrated_data["patient_id"].nunique()
    ),
    "duplicate_participant_ids": int(
        integrated_data["patient_id"]
        .duplicated()
        .sum()
    ),
    "columns_with_missing_values": int(
        (integrated_data.isna().sum() > 0).sum()
    ),
    "total_missing_values": int(
        integrated_data.isna().sum().sum()
    ),
    "total_infinite_values": (
        total_infinite_values
    ),
    "training_participants": len(train_data),
    "validation_participants": len(validation_data),
    "testing_participants": len(test_data),
    "train_validation_overlap": (
        train_validation_overlap
    ),
    "train_test_overlap": train_test_overlap,
    "validation_test_overlap": (
        validation_test_overlap
    )
}

validation_summary = pd.DataFrame(
    validation_results.items(),
    columns=["validation_check", "value"]
)

display(validation_summary)

,validation_check,value
0,integrated_rows,469
1,integrated_columns,241
2,unique_participants,469
3,duplicate_participant_ids,0
4,columns_with_missing_values,1
5,total_missing_values,1
6,total_infinite_values,0
7,training_participants,328
8,validation_participants,70
9,testing_participants,71


### Saving all required outputs

In [50]:
# Output file paths
wearable_output_file = (
    processed_folder
    / "wearable_features.csv"
)

integrated_output_file = (
    processed_folder
    / "integrated_participant_dataset.csv"
)

train_output_file = (
    processed_folder
    / "train_participant_dataset.csv"
)

validation_output_file = (
    processed_folder
    / "validation_participant_dataset.csv"
)

test_output_file = (
    processed_folder
    / "test_participant_dataset.csv"
)

validation_summary_file = (
    processed_folder
    / "data_integration_validation_summary.csv"
)

# Saving output datasets
wearable_features.to_csv(
    wearable_output_file,
    index=False
)

integrated_data.to_csv(
    integrated_output_file,
    index=False
)

train_data.to_csv(
    train_output_file,
    index=False
)

validation_data.to_csv(
    validation_output_file,
    index=False
)

test_data.to_csv(
    test_output_file,
    index=False
)

validation_summary.to_csv(
    validation_summary_file,
    index=False
)

print("All  output files saved successfully.")

All  output files saved successfully.


In [51]:
saved_files = {
    "Wearable features": wearable_output_file,
    "Integrated participant dataset": (
        integrated_output_file
    ),
    "Training dataset": train_output_file,
    "Validation dataset": validation_output_file,
    "Testing dataset": test_output_file,
    "Validation summary": validation_summary_file
}

for name, file_path in saved_files.items():
    print(
        f"{name}:",
        file_path.exists(),
        "—",
        file_path
    )

Wearable features: True — ..\data\processed\wearable_features.csv
Integrated participant dataset: True — ..\data\processed\integrated_participant_dataset.csv
Training dataset: True — ..\data\processed\train_participant_dataset.csv
Validation dataset: True — ..\data\processed\validation_participant_dataset.csv
Testing dataset: True — ..\data\processed\test_participant_dataset.csv
Validation summary: True — ..\data\processed\data_integration_validation_summary.csv


## Final Integration Summary

The participant-level integration workflow was completed successfully.

### Key Results

- 10,318 time-domain records were matched with 10,318 frequency-domain records.
- No wearable records were lost during the recording-level merge.
- Wearable outputs were aggregated to 469 unique participant records.
- The wearable table contains 183 participant-level wearable variables.
- Demographic, questionnaire, and wearable data were integrated into a dataset containing 469 participants and 242 columns.
- Every participant has exactly one final analytical record.
- No infinite values were detected.
- Ninety-nine missing values were identified across `age_at_diagnosis` and `height_cm` and were retained for leakage-safe downstream preprocessing.
- A stratified 70%/15%/15% participant-level split was created:
  - 328 training participants
  - 70 validation participants
  - 71 testing participants
- No participant overlap was detected across the three datasets.
- Target-related and identifier columns were identified as non-predictor variables to prevent model leakage.

### Exported Files

- `wearable_features.csv`
- `integrated_participant_dataset.csv`
- `train_participant_dataset.csv`
- `validation_participant_dataset.csv`
- `test_participant_dataset.csv`
- `data_integration_validation_summary.csv`